Batch-generate CFD wind field VTK files for all configured wind conditions.
Run this notebook **before** `training_data_user_guide.ipynb` to pre-populate the wind field library.

**Prerequisites:**
- Aerocae CFD tool must be open with the correct `.art` file loaded.
- Set `export_base_path` to the folder where Aerocae writes VTK exports.

In [ ]:
import sys
import os
import numpy as np
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import drone.disturbance_model
import drone.parameters
import drone.propeller
import drone.dynamics_state
import external_sim.wind_field_collector as wind_field_collector
import external_sim.cfd_wind_field_lookup.vtk_reader as vtk_reader

## Config

In [ ]:
# Hardcoded constants
WALL_ORIGIN_X = -0.5    # wall x-coordinate (m); negative = wall on the -x side
T_END = 5.0            # CFD run duration per wind condition (s)
DT = 0.01               # simulation timestep (s)

# Wind conditions to generate
wind_range_x = [-5.0, -2.0, 0.0]
wind_range_y = [0.0]
wind_range_z = [-4.0, -1.0, 0.0, 1.0, 4.0]

# Filesystem path where Aerocae writes VTK exports (used for skip-if-exists check)
export_base_path = Path(r"C:\Users\jiexu\Downloads\Aerocae Robotics - Geng 2025\export")

## Batch generation

In [ ]:
drone_params = drone.parameters.P600()
propeller_params = drone.propeller.apc_8x6
init_state = drone.dynamics_state.State()
n_steps = int(T_END / DT)

for wind_x in wind_range_x:
    for wind_y in wind_range_y:
        for wind_z in wind_range_z:
            wind_velocity = np.array([wind_x, wind_y, wind_z])
            folder_name = vtk_reader.get_wind_velocity_folder_name(wind_velocity, WALL_ORIGIN_X)

            if (export_base_path / folder_name).exists():
                print(f"[skip] {folder_name}")
                continue

            print(f"[run]  {folder_name}")
            disturbance = drone.disturbance_model.WindEffectNearWall(
                wall_origin=np.array([WALL_ORIGIN_X, 0.0, 0.0]),
                u_free=wind_velocity,
            )
            collector = wind_field_collector.WindFieldCollector(
                drone_params, propeller_params, disturbance, init_state, DT
            )
            for i in range(n_steps):
                collector.step(i * DT, None)
            collector.shutdown()
            print(f"[done] {folder_name}")